# 01 — Data Audit

## Objective
Perform an initial audit of the available NOVAres SecureHealth datasets to verify:

- dataset availability
- shapes and columns
- missing values
- duplicates
- key consistency across tables
- first business sanity checks

This notebook is the starting point for the analytical repository.

In [1]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Notebook cwd:", Path.cwd())
print("data/raw exists:", (PROJECT_ROOT / "data" / "raw").exists())
print("Files in data/raw:", os.listdir(PROJECT_ROOT / "data" / "raw"))

PROJECT_ROOT: C:\Users\dolivares\Desktop\novares-securehealth
Notebook cwd: C:\Users\dolivares\Desktop\novares-securehealth\notebooks
data/raw exists: True
Files in data/raw: ['claims_corrected.csv', 'dataset_summary.csv', 'insured_members.csv', 'member_year_features_corrected.csv', 'policies.csv', 'prospect_survey_synthetic.csv', 'providers.csv', 'provider_month_features.csv']


In [2]:
import pandas as pd
import numpy as np

from src.config import RAW_DIR
from src.data.load_data import (
    load_insured_members,
    load_policies,
    load_providers,
    load_claims,
    load_member_year_features,
    load_provider_month_features,
    load_prospect_survey,
    load_dataset_summary,
)

In [3]:
# Load datasets
members = load_insured_members()
policies = load_policies()
providers = load_providers()
claims = load_claims()
member_year = load_member_year_features()
provider_month = load_provider_month_features()
prospects = load_prospect_survey()
dataset_summary = load_dataset_summary()

datasets = {
    "insured_members": members,
    "policies": policies,
    "providers": providers,
    "claims": claims,
    "member_year_features": member_year,
    "provider_month_features": provider_month,
    "prospect_survey": prospects,
    "dataset_summary": dataset_summary,
}

In [4]:
summary_rows = []
for name, df in datasets.items():
    summary_rows.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

pd.DataFrame(summary_rows).sort_values("dataset").reset_index(drop=True)

,dataset,rows,columns
0,claims,44144,36
1,dataset_summary,7,3
2,insured_members,4000,32
3,member_year_features,8000,37
4,policies,4000,25
5,prospect_survey,7000,30
6,provider_month_features,4263,24
7,providers,180,17


In [5]:
# Columns by dataset
for name, df in datasets.items():
    print("=" * 80)
    print(name.upper())
    print("=" * 80)
    print(df.columns.tolist())
    print()

INSURED_MEMBERS
['member_id', 'policy_id', 'join_date', 'age', 'age_band', 'sex', 'region', 'city_tier', 'socioeconomic_band', 'employment_status', 'marital_status', 'dependents_n', 'family_complexity', 'smoker_flag', 'alcohol_risk_flag', 'physical_activity_level', 'bmi_group', 'chronic_condition_flag', 'chronic_condition_count', 'chronic_group', 'recurrent_medication_flag', 'prior_hospitalization_24m_flag', 'self_management_adherence', 'archetype', 'baseline_risk_score', 'utilization_propensity', 'acute_event_propensity', 'misuse_exposure_propensity', 'price_sensitivity', 'coverage_preference', 'network_preference', 'preferred_plan_type']

POLICIES
['policy_id', 'member_id', 'policy_start_date', 'policy_end_date', 'active_flag', 'plan_type', 'plan_tier', 'coverage_scope', 'provider_network_type', 'deductible_amount', 'copay_level', 'annual_coverage_limit', 'maternity_coverage_flag', 'pharmacy_coverage_flag', 'chronic_care_program_flag', 'premium_monthly', 'premium_annual', 'underwriti

In [6]:
# Preview first rows
for name, df in datasets.items():
    print("=" * 80)
    print(name.upper())
    display(df.head(3))

INSURED_MEMBERS


,member_id,policy_id,join_date,age,age_band,sex,region,city_tier,socioeconomic_band,employment_status,...,self_management_adherence,archetype,baseline_risk_score,utilization_propensity,acute_event_propensity,misuse_exposure_propensity,price_sensitivity,coverage_preference,network_preference,preferred_plan_type
0,MBR000001,POL000001,2022-09-24,47,45-54,M,San Pedro,Urban,Upper-Middle,Self-Employed,...,Low,Hyper-Utilizer / Misuse Risk,0.591,0.847,0.205,0.865,High,Balanced,Balanced,Managed Review
1,MBR000002,POL000002,2024-04-02,35,35-44,F,San Pedro,Metro,Lower-Middle,Employed,...,Low,Healthy Low User,0.156,0.104,0.074,0.085,Medium,Basic,Narrow,Essential
2,MBR000003,POL000003,2024-06-27,65,65+,M,Asunción,Urban,Middle,Self-Employed,...,High,Chronic Managed,0.728,0.776,0.223,0.056,Medium,Broad,Broad,Chronic Care


POLICIES


,policy_id,member_id,policy_start_date,policy_end_date,active_flag,plan_type,plan_tier,coverage_scope,provider_network_type,deductible_amount,...,premium_monthly,premium_annual,underwriting_load_factor,discount_factor,recommended_plan_flag,pricing_adequacy_ratio,renewal_flag,cancellation_flag,sales_channel,broker_id
0,POL000001,MBR000001,2022-09-24,2026-12-31,1,Managed Review,Mid,Full,Balanced,199.02,...,291.4,3496.8,1.139,0.092,1,0.851,1,0,Corporate,NaN
1,POL000002,MBR000002,2024-04-02,2026-12-31,1,Essential,Basic,Ambulatory,Narrow,597.15,...,198.7,2384.4,1.138,0.082,1,1.285,1,0,Direct,NaN
2,POL000003,MBR000003,2024-06-27,2026-12-31,1,Chronic Care,Premium,Full,Broad,196.63,...,430.1,5161.2,1.005,0.060,1,0.994,0,0,Direct,NaN


PROVIDERS


...

,provider_id,year_month,claims_n,members_unique_n,billed_total,approved_total,avg_claim_amount,median_claim_amount,p95_claim_amount,emergency_claims_n,...,dx_proc_mismatch_n,high_severity_share,suspicious_claim_ratio,provider_type,specialty_group,provider_archetype,cost_vs_peer_index,utilization_density_index,anomaly_score_provider,provider_alert_level
0,PRV00001,2024-01,13,13,2742.69,2392.39,210.976154,146.86,500.6120,1,...,0,0.077,0.000,Clinic,Pediatrics,Normal,0.465,1.0,0.010,Normal
1,PRV00001,2024-02,15,15,4396.21,4124.88,293.080667,214.67,753.0730,0,...,0,0.067,0.067,Clinic,Pediatrics,Normal,0.712,1.0,0.033,Normal
2,PRV00001,2024-03,22,22,4743.86,4029.39,215.630000,174.76,432.4795,3,...,0,0.136,0.091,Clinic,Pediatrics,Normal,0.460,1.0,0.042,Normal


PROSPECT_SURVEY


,prospect_id,age,age_band,sex,region,dependents_n,self_rated_health,chronic_condition_flag,chronic_condition_count,recurrent_medication_flag,...,maternity_interest_flag,pharmacy_need_flag,chronic_program_interest_flag,predicted_ground_truth_archetype,risk_tier,expected_utilization_band,recommended_plan,recommended_plan_tier,notes_driver_1,notes_driver_2
0,PRS000001,30,25-34,M,Alto Paraná,1,Good,0,0,0,...,0,0,0,Family Planner,Moderate,Medium,Family Plus,Mid,Family/dependent complexity,Preference for broader network
1,PRS000002,47,45-54,F,Central,1,Good,0,0,0,...,0,0,0,Preventive User,Moderate,Low,Standard,Mid,Moderate preventive utilization,Balanced coverage preference
2,PRS000003,18,18-24,F,Asunción,1,Excellent,0,0,0,...,0,0,0,Healthy Low User,Low,Very Low,Essential,Basic,Low recent utilization,Price-sensitive coverage preference


DATASET_SUMMARY


,dataset,rows,columns
0,insured_members.csv,4000,32
1,policies.csv,4000,25
2,claims.csv,44144,37


In [7]:
# Missing values audit
def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "column": df.columns,
        "missing_n": df.isna().sum().values,
        "missing_pct": (df.isna().sum().values / len(df)).round(4)
    })
    return out.sort_values(["missing_pct", "missing_n"], ascending=[False, False]).reset_index(drop=True)

for name, df in datasets.items():
    print("=" * 80)
    print(name.upper())
    display(missing_summary(df).head(15))

INSURED_MEMBERS


,column,missing_n,missing_pct
0,chronic_group,2852,0.713
1,member_id,0,0.000
2,policy_id,0,0.000
3,join_date,0,0.000
4,age,0,0.000
5,age_band,0,0.000
6,sex,0,0.000
7,region,0,0.000
8,city_tier,0,0.000
9,socioeconomic_band,0,0.000


POLICIES


,column,missing_n,missing_pct
0,broker_id,2256,0.564
1,policy_id,0,0.000
2,member_id,0,0.000
3,policy_start_date,0,0.000
4,policy_end_date,0,0.000
5,active_flag,0,0.000
6,plan_type,0,0.000
7,plan_tier,0,0.000
8,coverage_scope,0,0.000
9,provider_network_type,0,0.000


PROVIDERS


...

,column,missing_n,missing_pct
0,provider_id,0,0.0
1,year_month,0,0.0
2,claims_n,0,0.0
3,members_unique_n,0,0.0
4,billed_total,0,0.0
5,approved_total,0,0.0
6,avg_claim_amount,0,0.0
7,median_claim_amount,0,0.0
8,p95_claim_amount,0,0.0
9,emergency_claims_n,0,0.0


PROSPECT_SURVEY


,column,missing_n,missing_pct
0,prospect_id,0,0.0
1,age,0,0.0
2,age_band,0,0.0
3,sex,0,0.0
4,region,0,0.0
5,dependents_n,0,0.0
6,self_rated_health,0,0.0
7,chronic_condition_flag,0,0.0
8,chronic_condition_count,0,0.0
9,recurrent_medication_flag,0,0.0


DATASET_SUMMARY


,column,missing_n,missing_pct
0,dataset,0,0.0
1,rows,0,0.0
2,columns,0,0.0


In [8]:
# Duplicate rows audit
dup_rows = []
for name, df in datasets.items():
    dup_rows.append({
        "dataset": name,
        "duplicated_rows": int(df.duplicated().sum())
    })

pd.DataFrame(dup_rows).sort_values("duplicated_rows", ascending=False).reset_index(drop=True)

,dataset,duplicated_rows
0,insured_members,0
1,policies,0
2,providers,0
3,claims,0
4,member_year_features,0
5,provider_month_features,0
6,prospect_survey,0
7,dataset_summary,0


In [9]:
# Key integrity checks
key_checks = []

# insured_members
key_checks.append({
    "dataset": "insured_members",
    "key": "member_id",
    "unique_values": members["member_id"].nunique(),
    "rows": len(members),
    "is_unique": members["member_id"].nunique() == len(members)
})
key_checks.append({
    "dataset": "insured_members",
    "key": "policy_id",
    "unique_values": members["policy_id"].nunique(),
    "rows": len(members),
    "is_unique": members["policy_id"].nunique() == len(members)
})

# policies
key_checks.append({
    "dataset": "policies",
    "key": "policy_id",
    "unique_values": policies["policy_id"].nunique(),
    "rows": len(policies),
    "is_unique": policies["policy_id"].nunique() == len(policies)
})
key_checks.append({
    "dataset": "policies",
    "key": "member_id",
    "unique_values": policies["member_id"].nunique(),
    "rows": len(policies),
    "is_unique": policies["member_id"].nunique() == len(policies)
})

# providers
key_checks.append({
    "dataset": "providers",
    "key": "provider_id",
    "unique_values": providers["provider_id"].nunique(),
    "rows": len(providers),
    "is_unique": providers["provider_id"].nunique() == len(providers)
})

pd.DataFrame(key_checks)

,dataset,key,unique_values,rows,is_unique
0,insured_members,member_id,4000,4000,True
1,insured_members,policy_id,4000,4000,True
2,policies,policy_id,4000,4000,True
3,policies,member_id,4000,4000,True
4,providers,provider_id,180,180,True


In [10]:
print("member_id overlap between members and policies:",
      set(members["member_id"]) == set(policies["member_id"]))

print("policy_id overlap between members and policies:",
      set(members["policy_id"]) == set(policies["policy_id"]))

member_id overlap between members and policies: True
policy_id overlap between members and policies: True


In [11]:
# Data types overview
for name, df in datasets.items():
    print("=" * 80)
    print(name.upper())
    display(df.dtypes.reset_index().rename(columns={"index": "column", 0: "dtype"}))

INSURED_MEMBERS


,column,dtype
0,member_id,object
1,policy_id,object
2,join_date,object
3,age,int64
4,age_band,object
5,sex,object
6,region,object
7,city_tier,object
8,socioeconomic_band,object
9,employment_status,object


POLICIES


,column,dtype
0,policy_id,object
1,member_id,object
2,policy_start_date,object
3,policy_end_date,object
4,active_flag,int64
5,plan_type,object
6,plan_tier,object
7,coverage_scope,object
8,provider_network_type,object
9,deductible_amount,float64


PROVIDERS


...

,column,dtype
0,provider_id,object
1,year_month,object
2,claims_n,int64
3,members_unique_n,int64
4,billed_total,float64
5,approved_total,float64
6,avg_claim_amount,float64
7,median_claim_amount,float64
8,p95_claim_amount,float64
9,emergency_claims_n,int64


PROSPECT_SURVEY


,column,dtype
0,prospect_id,object
1,age,int64
2,age_band,object
3,sex,object
4,region,object
5,dependents_n,int64
6,self_rated_health,object
7,chronic_condition_flag,int64
8,chronic_condition_count,int64
9,recurrent_medication_flag,int64


DATASET_SUMMARY


,column,dtype
0,dataset,object
1,rows,int64
2,columns,int64


In [12]:
# First business sanity checks
print("insured_members age summary")
display(members["age"].describe())

print("policies premium_monthly summary")
display(policies["premium_monthly"].describe())

print("providers fraud_exposure_score summary")
display(providers["fraud_exposure_score"].describe())

insured_members age summary


count    4000.000000
mean       42.743000
std        14.264548
min        18.000000
25%        32.000000
50%        41.000000
75%        51.000000
max        80.000000
Name: age, dtype: float64

policies premium_monthly summary


count    4000.000000
mean      382.724860
std       194.615118
min       120.010000
25%       203.067500
50%       354.515000
75%       540.165000
max       897.500000
Name: premium_monthly, dtype: float64

providers fraud_exposure_score summary


count    180.000000
mean       0.280950
std        0.180704
min        0.120000
25%        0.183750
50%        0.226500
75%        0.300250
max        0.929000
Name: fraud_exposure_score, dtype: float64

In [13]:
print("insured_members age summary")
display(members["age"].describe())

print("policies premium_monthly summary")
display(policies["premium_monthly"].describe())

print("providers fraud_exposure_score summary")
display(providers["fraud_exposure_score"].describe())

insured_members age summary


count    4000.000000
mean       42.743000
std        14.264548
min        18.000000
25%        32.000000
50%        41.000000
75%        51.000000
max        80.000000
Name: age, dtype: float64

policies premium_monthly summary


count    4000.000000
mean      382.724860
std       194.615118
min       120.010000
25%       203.067500
50%       354.515000
75%       540.165000
max       897.500000
Name: premium_monthly, dtype: float64

providers fraud_exposure_score summary


count    180.000000
mean       0.280950
std        0.180704
min        0.120000
25%        0.183750
50%        0.226500
75%        0.300250
max        0.929000
Name: fraud_exposure_score, dtype: float64

In [14]:
sanity_flags = {
    "negative_ages": int((members["age"] < 0).sum()),
    "premium_monthly_non_positive": int((policies["premium_monthly"] <= 0).sum()),
    "premium_annual_non_positive": int((policies["premium_annual"] <= 0).sum()),
    "pricing_adequacy_ratio_non_positive": int((policies["pricing_adequacy_ratio"] <= 0).sum()),
    "fraud_exposure_score_outside_0_1": int(((providers["fraud_exposure_score"] < 0) | (providers["fraud_exposure_score"] > 1)).sum()),
}

pd.DataFrame(list(sanity_flags.items()), columns=["check", "n"])

,check,n
0,negative_ages,0
1,premium_monthly_non_positive,0
2,premium_annual_non_positive,0
3,pricing_adequacy_ratio_non_positive,0
4,fraud_exposure_score_outside_0_1,0


In [15]:
# Audit findings
audit_findings = [
    "Core dimensions available: members, policies, providers, claims, provider-month, member-year, prospects.",
    "Members and policies appear to be structurally one-to-one at current state.",
    "Providers table looks ready for enrichment of claims and anomaly analysis.",
    "Pricing-related variables already exist in policies and can support early pricing adequacy analysis.",
    "Archetype and synthetic propensity variables in insured_members provide a strong starting point for segmentation and risk scoring.",
]

for i, finding in enumerate(audit_findings, start=1):
    print(f"{i}. {finding}")

1. Core dimensions available: members, policies, providers, claims, provider-month, member-year, prospects.
2. Members and policies appear to be structurally one-to-one at current state.
3. Providers table looks ready for enrichment of claims and anomaly analysis.
4. Pricing-related variables already exist in policies and can support early pricing adequacy analysis.
5. Archetype and synthetic propensity variables in insured_members provide a strong starting point for segmentation and risk scoring.


## Next step
Proceed to:

- define entity relationships
- map variables by module
- specify analytical base tables

in `02_data_dictionary_and_join_plan.ipynb`.